# Week 10, Day 2 Lab — Real APIs, Error Handling & Retries
### From fake data to the live internet, in 2.5 hours

**Yesterday** your agent called fake, hardcoded functions. **Today every tool hits a real,
live service** — a real weather API, a real Wikipedia search, and a real (local) database.
Real services fail in ways fake ones never do, so today is also about making your agent
survive that.

**How to use this notebook:**
- **WRITE THIS** cells have only a spec/signature — you write the body.
- **TODO** cells ask you to modify or extend something already there.
- Plain cells are given — run them as-is.
- Stuck 5+ minutes on a WRITE THIS cell? The **Appendix** at the very bottom has solutions.

**Total time: 150 minutes**, including one 10-minute break.


---
## Section 0 — Setup

Same as yesterday: Gemini 2.5 Flash via Google's free `google-genai` SDK.

If you don't already have a key from Day 1: go to **https://aistudio.google.com/apikey**,
sign in, click **Create API key**, copy it. (If your Colab runtime restarted, you'll need to
re-enter it even if you used one yesterday — nothing is saved between sessions.)


In [1]:
!pip install -q google-genai


In [2]:
import os
import json
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")


Enter your Gemini API key: ··········


In [3]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.5-flash-lite"

test = client.models.generate_content(model=MODEL, contents="Say 'Day 2 lab is ready!'")
print(test.text)



Day 2 lab is ready!


---
## Section 1 — Recap: Fake vs. Real (15 minutes)

Yesterday's `get_weather` looked like this:

```python
fake_data = {"lahore": {"forecast": "sunny", "temp_c": 34}, ...}
```

Today, let's hit an **actual live weather API** directly — no agent, no tool schema yet,
just a plain HTTP request — so you can see what "real" costs us that "fake" didn't.


In [4]:
import requests

# Open-Meteo is a free, real weather API that needs NO API key at all.
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 31.5497, "longitude": 74.3436, "current_weather": True},
    timeout=5
)
response.raise_for_status()
print(response.json()["current_weather"])


{'time': '2026-09-02T14:30', 'interval': 900, 'temperature': 31.8, 'windspeed': 7.4, 'winddirection': 77, 'is_day': 0, 'weathercode': 0}


In [5]:
# Just swap in real coordinates, e.g. Lahore:
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 31.5497, "longitude": 74.3436, "current_weather": True},
    timeout=5
)
response.raise_for_status()
print(response.json()["current_weather"])

{'time': '2026-09-02T14:30', 'interval': 900, 'temperature': 31.8, 'windspeed': 7.4, 'winddirection': 77, 'is_day': 0, 'weathercode': 0}


---
## Section 2 — Worked Example: A Real Weather Tool, End to End

Here's the answer to "what about any city name?" — Open-Meteo also has a free **geocoding**
API (also no key needed) that turns a city name into coordinates. Our real tool will chain
**two** live API calls together: geocode the city, then fetch its weather.

This is fully worked for you — read it carefully, since Sections 4–5 ask you to write tools
with this exact same shape yourself.


In [6]:
# GIVEN — the real tool function: two chained live API calls
def get_weather(city: str) -> dict:
    """Look up real, current weather for any city name using free Open-Meteo APIs."""
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=5
    )
    geo.raise_for_status()
    results = geo.json().get("results")
    if not results:
        return {"error": f"Could not find a location matching '{city}'"}

    lat, lon = results[0]["latitude"], results[0]["longitude"]
    resolved_name = results[0]["name"]

    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": True},
        timeout=5
    )
    weather.raise_for_status()
    current = weather.json()["current_weather"]
    return {"city": resolved_name, "temp_c": current["temperature"], "windspeed": current["windspeed"]}


# Quick manual tests — try a real city and a nonsense one
print(get_weather("Lahore"))
print(get_weather("Xyzzyxutopia"))   # should return the {"error": ...} branch, not crash


{'city': 'Lahore', 'temp_c': 31.9, 'windspeed': 7.7}
{'error': "Could not find a location matching 'Xyzzyxutopia'"}


In [7]:
# GIVEN — the schema
get_weather_declaration = types.FunctionDeclaration(
    name="get_weather",
    description="Get real, current weather for any city name in the world.",
    parameters={
        "type": "object",
        "properties": {"city": {"type": "string", "description": "Any city name, e.g. 'Lahore' or 'Boston'"}},
        "required": ["city"]
    }
)

weather_tool = types.Tool(function_declarations=[get_weather_declaration])
config = types.GenerateContentConfig(tools=[weather_tool])
tool_registry = {"get_weather": get_weather}


### Bring back your `run_agent` loop from Day 1

Paste in the `run_agent` function you built yesterday (or use the version below if you'd
rather start from a clean copy — it's identical to the Day 1 Appendix solution).


In [8]:
# GIVEN — carried over from Day 1 (Appendix solution, in case you don't have your own copy handy)
def run_agent(user_query, config, tool_registry, max_steps=5, verbose=True):
    chat = client.chats.create(model=MODEL, config=config)
    response = chat.send_message(user_query)

    for step in range(max_steps):
        parts = response.candidates[0].content.parts
        function_calls = [p.function_call for p in parts if p.function_call]

        if not function_calls:
            if verbose:
                print(f"[Step {step+1}] Thought: I have enough info. Giving final answer.")
            return response.text

        function_response_parts = []
        for fc in function_calls:
            name = fc.name
            args = dict(fc.args)
            if verbose:
                print(f"[Step {step+1}] Action: {name}({args})")
            result = tool_registry[name](**args)
            if verbose:
                print(f"[Step {step+1}] Observation: {result}")
            function_response_parts.append(
                types.Part.from_function_response(name=name, response={"result": result})
            )

        response = chat.send_message(function_response_parts)

    return "Reached max steps without a final answer."


In [9]:
answer = run_agent("What's the weather like in Tokyo right now?", config=config, tool_registry=tool_registry)
print()
print("=" * 50)
print("FINAL ANSWER:", answer)


[Step 1] Action: get_weather({'city': 'Tokyo'})
[Step 1] Observation: {'city': 'Tokyo', 'temp_c': 25.2, 'windspeed': 2.7}
[Step 2] Thought: I have enough info. Giving final answer.

FINAL ANSWER: The weather in Tokyo right now is pleasant at 25.2°C with a gentle wind of 2.7 m/s.


**Notice:** this used yesterday's exact loop, unmodified — only the tool underneath changed
from fake to real. That's the whole point of building a generic loop: the tools are swappable.

**🔧 TODO Checkpoint (5 min):** ask about a city you're confident doesn't exist (misspell one
badly) and confirm your agent handles the `{"error": ...}` gracefully in its final answer,
rather than crashing.


In [10]:
# 🔧 TODO: ask about a city that doesn't exist, confirm graceful handling
answer = run_agent("What's the weather in Xyzzyxutopia?", config=config, tool_registry=tool_registry)
print(answer)


[Step 1] Action: get_weather({'city': 'Xyzzyxutopia'})
[Step 1] Observation: {'error': "Could not find a location matching 'Xyzzyxutopia'"}
[Step 2] Thought: I have enough info. Giving final answer.
I'm sorry, but I couldn't find a location named "Xyzzyxutopia." Could you please check the spelling or provide a different city?


---
## Section 3 — Error Handling & Retries (25 minutes)

Real APIs time out, rate-limit you, or occasionally the model passes an argument that doesn't
match what your function expects. Right now, if `get_weather` throws an uncaught exception
(e.g. a network timeout), your entire `run_agent` call crashes. Let's fix that at two levels.

### Level 1 — Safe argument handling

If the model ever calls a tool with the wrong argument names/types, `tool_registry[name](**args)`
raises a `TypeError` that currently kills the whole loop. Wrap it.


In [11]:
# WRITE THIS: safe_call_tool(name, args, tool_registry) -> dict
#
# Spec:
# - Try calling tool_registry[name](**args) and return its result
# - If it raises a TypeError (wrong/missing arguments), return {"error": f"Invalid arguments for {name}: {e}"}
# - If it raises ANY other Exception, return {"error": f"Unexpected error in {name}: {e}"}
# - Never let an exception escape this function

def safe_call_tool(name, args, tool_registry):
    try:
        return tool_registry[name](**args)
    except TypeError as e:
        return {"error": f"Invalid arguments for {name}: {e}"}
    except Exception as e:
        return {"error": f"Unexpected error in {name}: {e}"}


# Test: this should print an error dict, NOT raise an exception
print(safe_call_tool("get_weather", {"wrong_argument_name": "Lahore"}, tool_registry))

{'error': "Invalid arguments for get_weather: get_weather() got an unexpected keyword argument 'wrong_argument_name'"}


### Level 2 — Retry with exponential backoff

Network calls fail transiently sometimes — a retry a second later often just works. Write the
retry wrapper from the spec below


In [12]:
# WRITE THIS: call_with_retry(func, max_retries=3, **kwargs) -> result or error dict
#
# Spec:
# - Try calling func(**kwargs) and return its result if it succeeds
# - If it raises requests.exceptions.RequestException:
#     - if this was the last allowed attempt, return {"error": f"Failed after {max_retries} attempts: {e}"}
#     - otherwise, sleep for (2 ** attempt) seconds, then try again  (1s, 2s, 4s, ...)
# - Import `time` yourself

import time

def call_with_retry(func, max_retries=3, **kwargs):
    for attempt in range(max_retries):
        try:
            return func(**kwargs)
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                return {"error": f"Failed after {max_retries} attempts: {e}"}
            time.sleep(2 ** attempt)


# Test with a deliberately broken URL-based function to confirm retries + eventual failure message
def flaky_call():
    return requests.get("https://thisdomaindefinitelydoesnotexist12345.com", timeout=2).json()

print(call_with_retry(flaky_call, max_retries=2))


{'error': 'Failed after 2 attempts: HTTPSConnectionPool(host=\'thisdomaindefinitelydoesnotexist12345.com\', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7cf60e621450>: Failed to resolve \'thisdomaindefinitelydoesnotexist12345.com\' ([Errno -2] Name or service not known)"))'}


### Harden `run_agent` with both

**🔧 TODO:** modify the loop below (a copy of your `run_agent`) so that instead of calling
`tool_registry[name](**args)` directly, it uses your two new functions together — retry the
safe call. One line changes; the rest of the loop stays the same.


In [13]:
def run_agent_v2(user_query, config, tool_registry, max_steps=5, verbose=True):
    chat = client.chats.create(model=MODEL, config=config)
    response = chat.send_message(user_query)

    for step in range(max_steps):
        parts = response.candidates[0].content.parts
        function_calls = [p.function_call for p in parts if p.function_call]

        if not function_calls:
            if verbose:
                print(f"[Step {step+1}] Thought: I have enough info. Giving final answer.")
            return response.text

        function_response_parts = []
        for fc in function_calls:
            name = fc.name
            args = dict(fc.args)
            if verbose:
                print(f"[Step {step+1}] Action: {name}({args})")

            # 🔧 TODO: replace the line below with a call that combines
            # call_with_retry(...) AROUND safe_call_tool(...)

            def resilient_call(name=name, args=args, tool_registry=tool_registry):
                try:
                    return call_with_retry(lambda: tool_registry[name](**args))
                except TypeError as e:
                    return {"error": f"Invalid arguments for {name}: {e}"}
                except Exception as e:
                    return {"error": f"Unexpected error in {name}: {e}"}

            result = resilient_call()

            if verbose:
                print(f"[Step {step+1}] Observation: {result}")
            function_response_parts.append(
                types.Part.from_function_response(name=name, response={"result": result})
            )

        response = chat.send_message(function_response_parts)

    return "Reached max steps without a final answer."


In [14]:
# Test your hardened loop — should behave identically to run_agent for normal queries
answer = run_agent_v2("What's the weather in Lahore?", config=config, tool_registry=tool_registry)
print(answer)


[Step 1] Action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'temp_c': 31.9, 'windspeed': 7.7}
[Step 2] Thought: I have enough info. Giving final answer.
The current temperature in Lahore is 31.9°C with a wind speed of 7.7 km/h.


---
## Section 4 — Write a Real Search Tool: Wikipedia

Your turn to build a real-API tool completely from a spec, following the exact shape of
`get_weather` in Section 2. We'll use Wikipedia's public search API — real, free, no key.

**API reference (given):**
```
GET https://en.wikipedia.org/w/api.php
params: {"action": "query", "list": "search", "srsearch": <your query>,
         "format": "json", "srlimit": <num_results>}
```
A successful response looks like:
```json
{"query": {"search": [
    {"title": "Lahore", "snippet": "Lahore is the capital of..."},
    {"title": "Lahore Fort", "snippet": "..."}
]}}
```

**Spec for the function:**
- Signature: `search_wikipedia(query: str, num_results: int = 3) -> dict`
- Call the API above with `timeout=5` and `resp.raise_for_status()`
- Return `{"results": [{"title": ..., "snippet": ...}, ...]}` using the top `num_results` hits
- If there are zero results, return `{"results": []}` (not an error — an empty result is valid)


In [15]:
# WRITE THIS: search_wikipedia(query, num_results=3) -> dict, per the spec above

def search_wikipedia(query: str, num_results: int = 3) -> dict:
    resp = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
            "srlimit": num_results
        },
        headers={"User-Agent": "Week10Day2Lab/1.0 (student bootcamp assignment)"},
        timeout=5
    )
    resp.raise_for_status()
    hits = resp.json().get("query", {}).get("search", [])
    return {"results": [{"title": h["title"], "snippet": h["snippet"]} for h in hits[:num_results]]}


# Test it
print(search_wikipedia("Lahore Fort"))

{'results': [{'title': 'Lahore Fort', 'snippet': 'The <span class="searchmatch">Lahore</span> <span class="searchmatch">Fort</span> (Punjabi: شاہی قلعہ, romanised:\xa0Śā&#039;ī Qilā; Urdu: شاہی قلعہ, romanised:\xa0Shahi Qilah; lit.\u2009&#039;Royal <span class="searchmatch">Fort</span>&#039;) is a citadel in the walled interior'}, {'title': 'Sheesh Mahal (Lahore Fort)', 'snippet': 'located within the Shah Burj block at the north-western corner of the <span class="searchmatch">Lahore</span> <span class="searchmatch">Fort</span>, in <span class="searchmatch">Lahore</span>, Pakistan. It was constructed during the reign of Mughal Emperor'}, {'title': 'Lahore', 'snippet': 'shrines. <span class="searchmatch">Lahore</span> is also home to the <span class="searchmatch">Lahore</span> <span class="searchmatch">Fort</span> and Shalimar Gardens, both of which are UNESCO World Heritage Sites. The origin of <span class="searchmatch">Lahore&#039;s</span> name is unclear'}]}


Now write the schema (same pattern as `get_weather_declaration`), register it, and test the
agent end to end with a query that needs Wikipedia.


In [18]:
# WRITE THIS: FunctionDeclaration for search_wikipedia

search_wikipedia_declaration = types.FunctionDeclaration(
    name="search_wikipedia",
    description="Search Wikipedia for a topic and return the top matching article titles and snippets.",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search term, e.g. 'Lahore Fort'"},
            "num_results": {"type": "integer", "description": "Number of results to return (default 3)"}
        },
        "required": ["query"]
    }
)


In [19]:
# WRITE THIS: build a Tool + config with BOTH get_weather and search_wikipedia,
# and a tool_registry with both functions registered

multi_tool = types.Tool(function_declarations=[get_weather_declaration, search_wikipedia_declaration])
config = types.GenerateContentConfig(tools=[multi_tool])
tool_registry = {"get_weather": get_weather, "search_wikipedia": search_wikipedia}


In [20]:
answer = run_agent_v2("Tell me one interesting fact about Lahore from Wikipedia.",
                       config=config, tool_registry=tool_registry)
print(answer)


[Step 1] Action: search_wikipedia({'query': 'Lahore interesting facts history'})
[Step 1] Observation: {'results': [{'title': 'Culture of Lahore', 'snippet': 'the capital of the Punjab province of Pakistan. <span class="searchmatch">Lahore</span> has played an important role in Pakistani <span class="searchmatch">history</span>. It was in this city that Pakistan&#039;s independence'}, {'title': 'History of Punjab', 'snippet': 'Punjab, Pakistan in the <span class="searchmatch">Lahore</span> province of the Delhi Sultanate. The <span class="searchmatch">history</span> of the Sikh faith is closely associated with the <span class="searchmatch">history</span> of Punjab and the socio-political'}, {'title': 'Chand Bardai', 'snippet': 'ISBN\xa09788120801882. Rima Hooja (2006). A <span class="searchmatch">HISTORY</span> OF RAJASTHAN (PB). Rupa &amp; Company. pp.\xa0364–365. ISBN\xa0978-81-291-1501-0. <span class="searchmatch">Interestingly</span>, it is this version that today'}]}
[Step 2] Acti

**🔧 Checkpoint (3 min):** ask a question that should trigger a zero-result Wikipedia search
(a nonsense query) and confirm your agent handles an empty `results` list without crashing.


In [21]:
# 🔧 write a query likely to return zero Wikipedia results, then run it
answer = run_agent_v2("Wikipedia fact about asdkjfhqwoeiuraslkdjf1234nonsense?", config=config, tool_registry=tool_registry)
print(answer)


[Step 1] Action: search_wikipedia({'query': 'asdkjfhqwoeiuraslkdjf1234nonsense'})
[Step 1] Observation: {'results': []}
[Step 2] Thought: I have enough info. Giving final answer.
There are no Wikipedia articles or facts matching that exact nonsense string, as it appears to be a random sequence of letters and numbers! 

If you have a real topic, historical event, or person you'd like a Wikipedia fact about, let me know and I'd be happy to find one for you.


---
## Section 5 — Write a Real Database Tool + Test Full Chaining

Real production agents often query a real database. We'll create a small local SQLite
database (given — creating sample data isn't the learning goal here) and you'll write the
**query tool** yourself.


In [22]:
# GIVEN — set up a small local orders database with sample data
import sqlite3

conn = sqlite3.connect("orders.db")
conn.execute("DROP TABLE IF EXISTS orders")
conn.execute("CREATE TABLE orders (order_id TEXT, customer_id TEXT, status TEXT)")
sample_orders = [
    ("ORD-001", "CUST123", "shipped"),
    ("ORD-002", "CUST123", "processing"),
    ("ORD-003", "CUST456", "delivered"),
]
conn.executemany("INSERT INTO orders VALUES (?, ?, ?)", sample_orders)
conn.commit()
conn.close()
print("orders.db ready")


orders.db ready


**Spec for the tool:**
- Signature: `query_orders(customer_id: str) -> dict`
- Connect to `orders.db`, run a **parameterized** query (use `?` placeholders — never
  string-format `customer_id` directly into SQL) selecting `order_id, status` for that customer
- Return `{"orders": [{"order_id": ..., "status": ...}, ...]}`
- If there are no matching rows, return `{"orders": []}` (valid, not an error)


In [23]:
# WRITE THIS: query_orders(customer_id) -> dict, per the spec above

def query_orders(customer_id: str) -> dict:
    conn = sqlite3.connect("orders.db")
    cursor = conn.execute(
        "SELECT order_id, status FROM orders WHERE customer_id = ?",
        (customer_id,)
    )
    rows = cursor.fetchall()
    conn.close()
    return {"orders": [{"order_id": r[0], "status": r[1]} for r in rows]}


print(query_orders("CUST123"))   # should show 2 orders
print(query_orders("NOBODY"))    # should show an empty list, not crash

{'orders': [{'order_id': 'ORD-001', 'status': 'shipped'}, {'order_id': 'ORD-002', 'status': 'processing'}]}
{'orders': []}


In [24]:
# WRITE THIS: FunctionDeclaration for query_orders, then register all 3 tools together
# (get_weather, search_wikipedia, query_orders) into one Tool/config/tool_registry

query_orders_declaration = types.FunctionDeclaration(
    name="query_orders",
    description="Look up order status(es) for a given customer ID.",
    parameters={
        "type": "object",
        "properties": {
            "customer_id": {"type": "string", "description": "Customer ID, e.g. 'CUST123'"}
        },
        "required": ["customer_id"]
    }
)

all_tools = types.Tool(function_declarations=[
    get_weather_declaration, search_wikipedia_declaration, query_orders_declaration
])
config = types.GenerateContentConfig(tools=[all_tools])
tool_registry = {
    "get_weather": get_weather,
    "search_wikipedia": search_wikipedia,
    "query_orders": query_orders
}

### Test full 3-tool chaining


In [25]:
answer = run_agent_v2(
    "What's the weather in Lahore, tell me one fact about Lahore from Wikipedia, "
    "and check the order status for customer CUST123.",
    config=config, tool_registry=tool_registry
)
print()
print("=" * 50)
print("FINAL ANSWER:", answer)


[Step 1] Action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'temp_c': 31.9, 'windspeed': 7.7}
[Step 1] Action: search_wikipedia({'query': 'Lahore'})
[Step 1] Observation: {'results': [{'title': 'Lahore', 'snippet': '<span class="searchmatch">Lahore</span> is the capital and largest city of the Pakistani province of Punjab. It is the second-largest city in Pakistan, after Karachi, and 27th largest'}, {'title': 'Batwara 1947', 'snippet': 'produced by Aamir Khan under the banner of Aamir Khan Productions. Set in <span class="searchmatch">Lahore</span> against the backdrop of the 1947 Partition of British India and the division'}, {'title': 'Jis Lahore Nai Vekhya, O Jamya E Nai', 'snippet': 'Jis <span class="searchmatch">Lahore</span> Nai Vekhya, O Jamya E Nai (transl.\u2009 from Punjabi: Experiencing <span class="searchmatch">Lahore</span> is so fundamental to a fulfilling life that not seeing it is akin to not'}]}
[Step 1] Action: query_orders({'customer_id'

---
## Section 6 — Independent Challenge: Break It, Then Prove It Survives

You already wrapped `get_weather` in resilience via `run_agent_v2` — that wrapping applies
automatically to *every* tool in the registry, including the two you just wrote. Prove it.

**Task:** deliberately trigger a real failure in each tool and confirm the agent still returns
a sensible final answer instead of crashing.


In [26]:
# 🔧 Try to break each tool on purpose, one at a time, and confirm graceful handling:

# 1. A city that doesn't exist
print(run_agent_v2("Weather in Qwertyxyzville?", config=config, tool_registry=tool_registry, verbose=False))

# 2. A customer ID that doesn't exist
print(run_agent_v2("Order status for customer NOTAREALCUSTOMER?", config=config, tool_registry=tool_registry, verbose=False))

# 3. A Wikipedia query that returns nothing
print(run_agent_v2("Wikipedia fact about asdkjfhqwoeiuraslkdjf1234?", config=config, tool_registry=tool_registry, verbose=False))

I'm sorry, but I couldn't find a location named "Qwertyxyzville". Please check the spelling or try another city name!
No orders were found for customer ID `NOTAREALCUSTOMER`.
I searched Wikipedia for "asdkjfhqwoeiuraslkdjf1234", but no matching articles were found. It looks like a random string of characters! 

Let me know if you would like me to search for something else.


---
## Section 7 — Iteration Log Analysis
This is today's core lab task from the slides: **reading a trace and diagnosing a failure.**

First, generate your own real broken trace:


In [27]:
# GIVEN — run with verbose=True so every step prints, then deliberately confuse it
trace_answer = run_agent_v2(
    "What's the weather in Lahoreee and check order status for CUST999?",  # typo'd city + fake customer
    config=config, tool_registry=tool_registry, verbose=True
)
print()
print("FINAL ANSWER:", trace_answer)


[Step 1] Action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'temp_c': 31.7, 'windspeed': 7.5}
[Step 1] Action: query_orders({'customer_id': 'CUST999'})
[Step 1] Observation: {'orders': []}
[Step 2] Thought: I have enough info. Giving final answer.

FINAL ANSWER: The weather in Lahore is currently 31.7°C with a wind speed of 7.5 km/h. 

Regarding customer CUST999, there are no active orders found in the system.


Now analyze this **pre-recorded** trace from another (deliberately broken) run — a teammate's
agent that got stuck:

```
[Step 1] Action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'temp_c': 34, 'windspeed': 8.2}
[Step 2] Action: query_orders({'customer_id': 'CUST-123'})
[Step 2] Observation: {'orders': []}
[Step 3] Action: query_orders({'customer_id': 'CUST-123'})
[Step 3] Observation: {'orders': []}
[Step 4] Action: query_orders({'customer_id': 'CUST-123'})
[Step 4] Observation: {'orders': []}
Reached max steps without a final answer.
```

### Answer these in the markdown cell below (double-click to edit)
1. The weather lookup worked fine. What went wrong with the orders lookup, specifically?
   (Hint: compare `'CUST-123'` here against the sample data format from Section 5.)
2. Why did the agent call `query_orders` three times with the *exact same* (wrong) argument
   instead of trying something different or giving up sooner?
3. Whose fault is this, really — the model's, the tool's, or the data's? What's the one-line
   fix?


 # **Your analysis:**

1.What went wrong with the orders lookup?
The agent called query_orders({'customer_id': 'CUST-123'}), but the sample data in Section 5 stores IDs without a hyphen ('CUST123'). 'CUST-123' != 'CUST123', so the SQL WHERE customer_id = ? matches nothing — a valid, empty result, not a crash. The lookup "worked" mechanically but returned no rows because of a formatting mismatch between what the model guessed and what's actually in the database.

2.Why did it retry the exact same wrong argument three times?
Nothing in the tool's response told the model why the result was empty — {"orders": []} looks identical whether the customer doesn't exist, the ID format is wrong, or there's genuinely no order history. With no error signal or hint to change its approach, the model's only available move was to try again, and since it had no new information, it repeated the same call rather than exploring alternatives (e.g., stripping the hyphen). After the third identical failure it simply ran out of steps.

3.Whose fault is this, and what's the one-line fix?
It's a tool design problem, not the model's or the data's — the tool is technically correct (empty result is a valid, non-error state per spec), but it gives the model no actionable signal to course-correct. One-line fix: normalize the ID inside query_orders before querying (e.g. customer_id = customer_id.replace("-", "").upper()), so format variance from the model doesn't produce silent empty results.


